# Chemprop v2 — Experiment 0: Negative Control (No Stereochemistry)

This notebook establishes the negative control baseline for the Chemprop chirality
classification experiments. All stereochemical information is stripped from the
molecular graphs via `ignore_stereo=True`, making both enantiomers of each compound
identical from the MPNN's perspective.

**Expected result:** all three targets (`R/S_class`, `@/@@_class`, `F/L_class`)
should be predicted at chance i.e., ~50% AUROC, ~50% accuracy, MCC ≈ 0.

**Conditions in this notebook:**

| Condition | Type | Description |
|---|---|---|
| `exp_0_neg_control_no_stereo_singletask` | Single-task | Independent model per target, stereo stripped |
| `exp_0_neg_control_no_stereo_multitask` | Multi-task | Joint model for all 3 targets, stereo stripped |

**Total fits:** 1 condition × 3 targets × 5 folds (singletask) + 5 folds (multitask) = 20

Part of series: `3_RF_morgan_count.ipynb` → **`4_chemprop_exp_0.ipynb`** → `4_chemprop_exp_1.ipynb`

## Imports

In [1]:
import gc
import time
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import torch

from lightning import pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import (
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score,
    f1_score,
    average_precision_score,
)

from chemprop import data, featurizers, models, nn

## Configuration

Chemprop's MoleculeDataset uses PyTorch's DataLoader object
- https://github.com/chemprop/chemprop/blob/7033cac6ebc1a66f885705c7fa3e2cb960f96e49/chemprop/data/datasets.py#L189

- By default, `num_workers=0`, which means data is loaded sequentially inside the main
  execution process.
- Setting `num_workers=1` forces Python to spawn a separate background process, serialize
  the data, transfer it across shared memory, and deserialize it back. This is not efficient
  given the size of the dataset here.

In [2]:
INPUT_PATH  = 'data/class_all.csv'
FOLDS_PATH  = 'data/cmrt_folds.npz'
NUM_WORKERS = 0      # set >0 if multiprocessing is available
MAX_EPOCHS  = 50     # upper bound; EarlyStopping will typically stop sooner
PATIENCE    = 10     # early stopping patience
BATCH_SIZE  = 64
SEED        = 42

pl.seed_everything(SEED, workers=True)

Seed set to 42


42

## Load data and splits

In [3]:
df = pd.read_csv(INPUT_PATH, index_col=0)
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (3858, 6)


,SMILES,SMILES_opp,TR/TE,F/L_class,@/@@_class,R/S_class
0,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,TE,F,@,S
1,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,TE,L,@@,R
2,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,TE,F,@,S
3,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,TE,L,@@,R
4,C=C(C(C)=O)[C@@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,C=C(C(C)=O)[C@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,TE,F,@@,R


In [4]:
def load_folds(path: str) -> list[dict[str, np.ndarray]]:
    """
    Load pre-computed splits from a .npz file.

    Args:
        path: Path to .npz file saved by save_folds() in notebook 2.

    Returns:
        List of dicts with keys 'train', 'val', 'test' as numpy arrays
        of molecule-level integer indices into the full dataset.
    """
    archive = np.load(path)
    fold_indices = sorted(set(
        int(k.split('_')[0].replace('fold', '')) for k in archive.files))
    return [
        {
            'train': archive[f'fold{i}_train'],
            'val':   archive[f'fold{i}_val'],
            'test':  archive[f'fold{i}_test'],
        }
        for i in fold_indices
    ]

In [5]:
mol_folds = load_folds(FOLDS_PATH)
print(f'Loaded {len(mol_folds)} folds')
for i, fold in enumerate(mol_folds):
    n_tr = len(fold['train'])
    n_v  = len(fold['val'])
    n_te = len(fold['test'])
    print(f'  Fold {i}: train={n_tr:,}  val={n_v:,}  test={n_te:,}')

Loaded 5 folds
  Fold 0: train=3,478  val=190  test=190
  Fold 1: train=3,478  val=190  test=190
  Fold 2: train=3,478  val=190  test=190
  Fold 3: train=3,478  val=190  test=190
  Fold 4: train=3,478  val=190  test=190


## Target variables and label encoding

All three targets are binary and perfectly balanced (50/50). Chance baseline: 50%.

In [6]:
TARGET_LABEL_MAPS = {
    'R/S_class':  {'R': 0, 'S': 1},
    '@/@@_class': {'@': 0, '@@': 1},
    'F/L_class':  {'F': 0, 'L': 1},
}
SINGLE_TARGETS = list(TARGET_LABEL_MAPS.keys())

for col, lmap in TARGET_LABEL_MAPS.items():
    df[f'label_{col}'] = df[col].map(lmap)

print('Class distributions (all should be 50/50):')
for col in SINGLE_TARGETS:
    vc = df[f'label_{col}'].value_counts(normalize=True)
    print(f'  {col}: {vc.to_dict()}')

Class distributions (all should be 50/50):
  R/S_class: {1: 0.5, 0: 0.5}
  @/@@_class: {0: 0.5, 1: 0.5}
  F/L_class: {0: 0.5, 1: 0.5}


## Featurizer: negative control — stereochemistry stripped

`SimpleMoleculeMolGraphFeaturizer` is used in both conditions, but `ignore_stereo=True`
is passed to `MoleculeDatapoint.from_smi()` for the negative control.

**What `ignore_stereo=True` does:** before building the RDKit mol object, all `@/@@`
annotations are stripped from the SMILES string. The resulting molecular graphs are
*identical for both enantiomers* — the MPNN has no information to distinguish them.

**Expected result:** all three targets should be predicted at chance (~50% AUROC,
~50% accuracy, MCC ≈ 0). Any model that scores above chance on this condition has
found a confound, not a stereochemical signal.

**Multi-task negative control (`exp_0_neg_control_no_stereo_multitask`):** the same
stereo-stripped datapoints are used, with `BinaryClassificationFFN(n_tasks=3)`. This
confirms the null result holds when all three targets are predicted jointly.

In [7]:
_feat_inspect = featurizers.SimpleMoleculeMolGraphFeaturizer()
from rdkit import Chem
_mol = Chem.MolFromSmiles('Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2')
_graph = _feat_inspect(_mol)
print(f'Atom feature dim: {_graph.V.shape[1]}')
print(f'Bond feature dim: {_graph.E.shape[1]}')
print('ChiralTag is a one-hot feature within the atom feature vector.')
print('Values: UNSPECIFIED=0, CW(@@)=1, CCW(@)=2')
print('With ignore_stereo=True this slot is always UNSPECIFIED for every atom.')

Atom feature dim: 72
Bond feature dim: 14
ChiralTag is a one-hot feature within the atom feature vector.
Values: UNSPECIFIED=0, CW(@@)=1, CCW(@)=2
With ignore_stereo=True this slot is always UNSPECIFIED for every atom.


In [8]:
# ignore_stereo=True: strips all @/@@ annotations before graph construction
# Both enantiomers produce the identical molecular graph — the negative control
all_data_no_stereo = [
    data.MoleculeDatapoint.from_smi(smi, ignore_stereo=True)
    for smi in df['SMILES']
]

print(f'Built {len(all_data_no_stereo):,} no-stereo datapoints (negative control)')

Built 3,858 no-stereo datapoints (negative control)


## Model and trainer builder functions

### Message Passing
A `Message passing` constructs molecular graphs using message passing to learn node-level hidden representations.

Options are `mp = nn.BondMessagePassing()` or `mp = nn.AtomMessagePassing()`

### Aggregation
An `Aggregation` constructs a graph-level representation from the set of
node-level representations after message passing. The permutation-invariant mean
is the standard Chemprop aggregation and is precisely what limits chirality signal
survival.

Available options can be found in ` nn.agg.AggregationRegistry` https://github.com/chemprop/chemprop/blob/main/chemprop/nn/agg.py#L106
- `agg = nn.MeanAggregation()`
- `agg = nn.SumAggregation()`
- `agg = nn.NormAggregation()`  by default, it divides the sum by 100

### Feed-Forward Network (FFN)

A `FFN` takes the aggregated representations and make target predictions.

Available options can be found in `nn.PredictorRegistry`.

For regression:
- `ffn = nn.RegressionFFN()`
- `ffn = nn.MveFFN()`
- `ffn = nn.EvidentialFFN()`

For classification:
- `ffn = nn.BinaryClassificationFFN()`
- `ffn = nn.BinaryDirichletFFN()`
- `ffn = nn.MulticlassClassificationFFN()`  input molecule can only belong to 1 of the n targets 
- `ffn = nn.MulticlassDirichletFFN()`

For spectral:
- `ffn = nn.SpectralFFN()` # will be available in future version


`nn.BinaryClassificationFFN(n_tasks=1)` for single-task; `n_tasks=3` for multi-task.
Both use BCELoss internally and apply sigmoid to produce per-task probabilities.

In [9]:
def build_mpnn(n_tasks: int = 1) -> models.MPNN:
    """
    Build a fresh Chemprop v2 MPNN for binary classification.

    A new model is instantiated for every (condition, target, fold) to
    ensure no weight sharing across runs.

    For the BinaryClassificationFFN, n_targets is hardcoded to 1 and the default loss is BCELoss
    https://github.com/chemprop/chemprop/blob/7033cac6ebc1a66f885705c7fa3e2cb960f96e49/chemprop/nn/predictors.py#L236

    Passing n_tasks=3 means the output dimension becomes a tensor of shape (batch_size, 3).
    A .sigmoid() activation is applied to the entire tensor so each of the 3 targets is evaluated
    independently and gets its own distinct probability between 0 and 1.

    Args:
        n_tasks: Number of binary classification outputs. 1 for single-task;
            3 for the multi-task condition predicting all targets jointly.

    Returns:
        Configured chemprop.models.MPNN ready for training.
    """
    mp  = nn.BondMessagePassing()
    agg = nn.MeanAggregation()
    ffn = nn.BinaryClassificationFFN(n_tasks=n_tasks)  
    metric_list = [
        nn.metrics.BinaryAUROC(),    # primary — used for EarlyStopping
        nn.metrics.BinaryAUPRC(),
        nn.metrics.BinaryAccuracy(),
        nn.metrics.BinaryF1Score(),
        # MCC is not natively available in Chemprop; computed post-hoc with sklearn
    ]
    return models.MPNN(mp, agg, ffn, batch_norm=False, metrics=metric_list)


def build_trainer(fold_idx: int, condition_name: str, target_col: str) -> pl.Trainer:
    """
    Build a Lightning Trainer with EarlyStopping and ModelCheckpoint.

    EarlyStopping monitors val/roc (BinaryAUROC) and stops if it does not improve
    for PATIENCE consecutive epochs.

    Args:
        fold_idx: The current fold number (used for checkpoint naming).
        condition_name: The featurization condition (used for checkpoint naming).
        target_col: The target being predicted (used for checkpoint naming).

    Returns:
        Configured pl.Trainer.
    """
    # must use `monitor='val/roc'`, as logged by Chemprop
    # monitor='val_BinaryAUROC' causes RuntimeError: Early stopping conditioned on metric
    # `val_BinaryAUROC` which is not available. Use any of: `train_loss`, `val/roc`, etc.
    early_stop = EarlyStopping(
        monitor='val/roc',
        patience=PATIENCE,
        mode='max',
        verbose=False,
    )

    # Add ModelCheckpoint to save the best model
    checkpoint_callback = ModelCheckpoint(
        dirpath='checkpoints/',
        filename=f"{condition_name}_{target_col.replace('/', '_')}_fold{fold_idx}_best",
        monitor='val/roc',
        mode='max',
        save_top_k=1, # Only save the best one
        verbose=False,
    )

    return pl.Trainer(
        logger=False,
        enable_checkpointing=True,
        enable_progress_bar=False,   # suppresses per-batch output in loops
        accelerator='auto',
        devices=1,
        max_epochs=MAX_EPOCHS,
        callbacks=[early_stop, checkpoint_callback],
    )

## `evaluate_chemprop` helper

Loads the best checkpoint and computes all metrics for a single loader/split. MCC is computed post-hoc via sklearn (not available natively in Chemprop v2).

Because ModelCheckpoint defines a unique filename based on the current loop variables the Trainer looks internally at the specific ModelCheckpoint instance that was just active during its fit() run, gets the path to that specific file, and loads it. It has no awareness of the checkpoints saved by prior instances of the loop.

In [10]:
def evaluate_chemprop(
    trainer: pl.Trainer,
    mpnn: models.MPNN,
    loader,
    y_true: np.ndarray,
) -> dict:
    """
    Run prediction and compute all metrics for one fold/split.

    MCC is not natively available in Chemprop v2 metrics, so it is computed
    post-hoc using sklearn at threshold 0.5, consistent with the RF notebooks.

    Args:
        trainer: Fitted pl.Trainer.
        mpnn: Fitted MPNN model.
        loader: DataLoader to run predictions on.
        y_true: 1D numpy array of true binary labels.

    Returns:
        Dict mapping metric name to scalar value.
    """
    # ckpt_path='best' loads the best checkpoint saved by ModelCheckpoint.
    # weights_only=False is required to load Chemprop's custom metric objects (BinaryAUROC etc.) from the PyTorch Lightning checkpoint without an UnpicklingError.
    preds = trainer.predict(mpnn, loader, ckpt_path='best', weights_only=False)
    probs = torch.cat(preds).squeeze().numpy()
    preds_bin = (probs > 0.5).astype(int)
    return {
        'AUROC':    roc_auc_score(y_true, probs),
        'MCC':      matthews_corrcoef(y_true, preds_bin),
        'Accuracy': accuracy_score(y_true, preds_bin),
        'F1':       f1_score(y_true, preds_bin),
        'AUPRC':    average_precision_score(y_true, probs),
    }

## Single-task training loop — Exp 0

One condition × 3 targets × 5 folds = **15 total fits**

The `val` split drives EarlyStopping and is not used for evaluation.
Train, val, and test scores are all appended to `all_results` for completeness,
but only test scores are printed during the loop.

In [11]:
# Only one condition in this notebook: the negative control
SINGLE_TASK_CONDITIONS = {
    'exp_0_neg_control_no_stereo_singletask': all_data_no_stereo,
}

# Total single-task fits: 1 condition x 3 targets x 5 folds = 15
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
all_results = []

for condition_name, all_dpoints in SINGLE_TASK_CONDITIONS.items():
    START_TIME = time.time()
    print(f"\n{'#'*70}")
    print(f'Condition: {condition_name}')
    if 'neg_control' in condition_name:
        print('  Negative control: stereo stripped, enantiomers are identical graphs')
        print('  Expected: all targets at chance (~50% accuracy, AUROC ~0.50)')
    print(f"{'#'*70}")

    for target_col in SINGLE_TARGETS:
        label_col = f'label_{target_col}'
        print(f"\n  Target: {target_col}\n  {'-'*60}")

        for fold_idx, fold in enumerate(mol_folds):
            # Slice datapoints by fold index
            tr_pts = [all_dpoints[i] for i in fold['train']]
            va_pts = [all_dpoints[i] for i in fold['val']]
            te_pts = [all_dpoints[i] for i in fold['test']]

            y_train = df[label_col].iloc[fold['train']].values.reshape(-1, 1)
            y_val   = df[label_col].iloc[fold['val']].values.reshape(-1, 1)
            y_test  = df[label_col].iloc[fold['test']].values.reshape(-1, 1)

            # Attach labels to datapoints for this run
            for dp, yi in zip(tr_pts, y_train): dp.y = yi
            for dp, yi in zip(va_pts, y_val):   dp.y = yi
            for dp, yi in zip(te_pts, y_test):  dp.y = yi

            train_dset = data.MoleculeDataset(tr_pts, featurizer)
            val_dset   = data.MoleculeDataset(va_pts, featurizer)
            test_dset  = data.MoleculeDataset(te_pts, featurizer)

            train_loader = data.build_dataloader(
                train_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=True)
            val_loader   = data.build_dataloader(
                val_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=False)
            test_loader  = data.build_dataloader(
                test_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=False)

            mpnn = build_mpnn(n_tasks=1)
            trainer = build_trainer(fold_idx, condition_name, target_col)
            trainer.fit(mpnn, train_loader, val_loader)

            # Evaluate on all three splits
            splits_to_eval = [
                ('train', train_loader, y_train.ravel()),
                ('val', val_loader, y_val.ravel()),
                ('test', test_loader, y_test.ravel())
            ]

            for split_name, loader, y_true in splits_to_eval:
                metrics = evaluate_chemprop(trainer, mpnn, loader, y_true)
                metrics.update({
                    'fold':          fold_idx,
                    'model':         'chemprop',
                    'featurization': condition_name,
                    'target':        target_col,
                    'split':         split_name,
                    'stopped_epoch': trainer.current_epoch,
                })
                all_results.append(metrics)

                if split_name == 'test':
                    print(f'  Elapsed fold {fold_idx}: {time.time() - START_TIME:.1f}s')
                    print(
                        f'  fold {fold_idx} (test) | '
                        f"AUROC={metrics['AUROC']:.3f}  "
                        f"MCC={metrics['MCC']:.3f}  "
                        f"Acc={metrics['Accuracy']:.3f}  "
                        f"stopped_epoch={metrics['stopped_epoch']}"
                    )

            # free up memory
            # delete the Python references to the large objects
            del mpnn, trainer
            # force Python's Garbage Collector to immediately clean up RAM
            gc.collect()
            # Dynamically clear the hardware accelerator cache
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            elif torch.backends.mps.is_available():
                torch.mps.empty_cache()

    print(f'Elapsed time for {condition_name}: {time.time() - START_TIME:.1f} seconds')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.



######################################################################
Condition: exp_0_neg_control_no_stereo_singletask
  Negative control: stereo stripped, enantiomers are identical graphs
  Expected: all targets at chance (~50% accuracy, AUROC ~0.50)
######################################################################

  Target: R/S_class
  ------------------------------------------------------------


/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_0_neg_control_no_stereo_singletask_F_L_class_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_0_neg_control_no_stereo_singletask_F_L_class_fold4_best.ckpt
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_0_neg_control_no_stereo_singletask_F_L_class_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_0_neg_control_no_stereo_singletask_F_L_class_fold4_best.ckpt
/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Restoring states from the checkpoint path at /Users/k

  Elapsed fold 4: 1095.5s
  fold 4 (test) | AUROC=0.500  MCC=0.000  Acc=0.500  stopped_epoch=13
Elapsed time for exp_0_neg_control_no_stereo_singletask: 1095.7 seconds


## Multi-task training loop — Exp 0

`BinaryClassificationFFN(n_tasks=3)` with stereo-stripped datapoints. The model is trained with a combined binary cross-entropy loss averaged across the three tasks.
5 folds = **5 total fits**.

If the singletask negative control is correct, this should also be at chance.

In [12]:
# ── Exp 0 multitask negative control ─────────────────────────────────────────
# Same stereo-stripped datapoints; joint prediction of all 3 targets.
# Total fits: 5 folds

print(f"\n{'#'*70}")
MULTITASK_CONDITION = 'exp_0_neg_control_no_stereo_multitask'
print(f'Condition: {MULTITASK_CONDITION}')
print('  Negative control: stereo stripped, all 3 targets predicted jointly')
print('  Expected: all targets at chance (~50% accuracy, AUROC ~0.50)')
print(f"{'#'*70}")

for fold_idx, fold in enumerate(mol_folds):
    START_TIME = time.time()
    print(f"\n  Fold {fold_idx}  |  "
          f"train={len(fold['train']):,}  "
          f"val={len(fold['val']):,}  "
          f"test={len(fold['test']):,}")

    tr_pts = [all_data_no_stereo[i] for i in fold['train']]
    va_pts = [all_data_no_stereo[i] for i in fold['val']]
    te_pts = [all_data_no_stereo[i] for i in fold['test']]

    label_cols = [f'label_{t}' for t in SINGLE_TARGETS]
    y_train = df[label_cols].iloc[fold['train']].values   # (n_train, 3)
    y_val   = df[label_cols].iloc[fold['val']].values
    y_test  = df[label_cols].iloc[fold['test']].values

    for dp, yi in zip(tr_pts, y_train): dp.y = yi
    for dp, yi in zip(va_pts, y_val):   dp.y = yi
    for dp, yi in zip(te_pts, y_test):  dp.y = yi

    train_dset = data.MoleculeDataset(tr_pts, featurizer)
    val_dset   = data.MoleculeDataset(va_pts, featurizer)
    test_dset  = data.MoleculeDataset(te_pts, featurizer)

    train_loader = data.build_dataloader(
        train_dset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True)
    val_loader   = data.build_dataloader(
        val_dset,   batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)
    test_loader  = data.build_dataloader(
        test_dset,  batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)

    mpnn    = build_mpnn(n_tasks=3)
    trainer = build_trainer(fold_idx, MULTITASK_CONDITION, 'all_targets')
    trainer.fit(mpnn, train_loader, val_loader)

    splits_to_eval = [
        ('train', train_loader, y_train),
        ('val',   val_loader,   y_val),
        ('test',  test_loader,  y_test),
    ]

    for split_name, loader, y_true_all in splits_to_eval:
        preds     = trainer.predict(mpnn, loader, ckpt_path='best', weights_only=False)
        all_probs = torch.cat(preds).numpy()   # shape (n, 3)

        for task_idx, target_col in enumerate(SINGLE_TARGETS):
            probs     = all_probs[:, task_idx]
            y_true    = y_true_all[:, task_idx]
            preds_bin = (probs > 0.5).astype(int)
            metrics = {
                'AUROC':    roc_auc_score(y_true, probs),
                'MCC':      matthews_corrcoef(y_true, preds_bin),
                'Accuracy': accuracy_score(y_true, preds_bin),
                'F1':       f1_score(y_true, preds_bin),
                'AUPRC':    average_precision_score(y_true, probs),
                'fold':          fold_idx,
                'model':         'chemprop',
                'featurization': MULTITASK_CONDITION,
                'target':        target_col,
                'split':         split_name,
                'stopped_epoch': trainer.current_epoch,
            }
            all_results.append(metrics)

            if split_name == 'test':
                print(
                    f"  {target_col} (test): "
                    f"AUROC={metrics['AUROC']:.3f}  "
                    f"MCC={metrics['MCC']:.3f}  "
                    f"Acc={metrics['Accuracy']:.3f}"
                )

    print(f'Elapsed time for {MULTITASK_CONDITION} fold {fold_idx}: {time.time() - START_TIME:.1f} seconds')

    del mpnn, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.



######################################################################
Condition: exp_0_neg_control_no_stereo_multitask
  Negative control: stereo stripped, all 3 targets predicted jointly
  Expected: all targets at chance (~50% accuracy, AUROC ~0.50)
######################################################################

  Fold 0  |  train=3,478  val=190  test=190


/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 91.2 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.
/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  227 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 91.2 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 318 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 318 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_0_neg_control_no_stereo_multitask_all_targets_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_0_neg_control_no_stereo_multitask_all_targets_fold4_best.ckpt
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_0_neg_control_no_stereo_multitask_all_targets_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_0_neg_control_no_stereo_multitask_all_targets_fold4_best.ckpt
/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Restoring states from the checkpoint path at /Use

  R/S_class (test): AUROC=0.500  MCC=0.000  Acc=0.500
  @/@@_class (test): AUROC=0.500  MCC=0.000  Acc=0.500
  F/L_class (test): AUROC=0.500  MCC=0.000  Acc=0.500
Elapsed time for exp_0_neg_control_no_stereo_multitask fold 4: 76.3 seconds


## Summary across folds

F1 is sensitive to the threshold and to which class the model arbitrarily assigns when features are tied, so minor implementation-level randomness can produce small fluctuations even when the model has zero discriminative power.

In [13]:
results_df = pd.DataFrame(all_results)
metric_cols  = ['AUROC', 'MCC', 'Accuracy', 'F1', 'AUPRC']
target_order = ['R/S_class', '@/@@_class', 'F/L_class']

# dynamically grab the conditions that were actually run (future-proofs for any change in experiment name or additional experiments)
# condition_order = sorted(results_df['featurization'].unique())
condition_order = [
    'exp_0_neg_control_no_stereo_singletask',
    'exp_0_neg_control_no_stereo_multitask',
]

# Filter to only the test set for the printed summary 
test_results = results_df[results_df['split'] == 'test']
summary = (
    test_results
    .groupby(['featurization', 'target'])[metric_cols]
    .agg(['mean', 'std'])
    .round(4)
)
# Reindex to ensure targets print in the preferred order
summary = summary.reindex([(c, t) for c in condition_order for t in target_order])
print('Test Set Performance (Mean ± std across 5 folds):')
print(summary.to_string())

Test Set Performance (Mean ± std across 5 folds):
                                                  AUROC          MCC      Accuracy           F1           AUPRC        
                                                   mean     std mean  std     mean  std    mean     std    mean     std
featurization                          target                                                                          
exp_0_neg_control_no_stereo_singletask R/S_class    0.5  0.0000  0.0  0.0      0.5  0.0  0.3815  0.3503  0.5001  0.0002
                                       @/@@_class   0.5  0.0000  0.0  0.0      0.5  0.0  0.6667  0.0000  0.5000  0.0000
                                       F/L_class    0.5  0.0000  0.0  0.0      0.5  0.0  0.0000  0.0000  0.5000  0.0000
exp_0_neg_control_no_stereo_multitask  R/S_class    0.5  0.0001  0.0  0.0      0.5  0.0  0.3259  0.3337  0.5000  0.0001
                                       @/@@_class   0.5  0.0001  0.0  0.0      0.5  0.0  0.4041  0.3596  0.500

## Sanity checks

**Negative control (`no_stereo`):** all targets should be at chance (~50% accuracy,
AUROC ~0.50, MCC ~0). Enantiomers are identical graphs so the model has no basis
to distinguish them.

In [14]:
# ── Negative control sanity check ────────────────────────────────────────────
# All conditions/targets should be at chance. Flag anything above chance.
print(f"{'Target':<15} {'Condition':<45} {'AUROC':>7} {'MCC':>7} {'Acc':>7}  Note")
print('-' * 90)
for target in target_order:
    for cond in condition_order:
        # must filter for 'test' split so we don't average in train/val scores
        s = results_df[
            (results_df['target'] == target) &
            (results_df['featurization'] == cond) &
            (results_df['split'] == 'test')
        ]
        if s.empty:
            continue
        auroc = s['AUROC'].mean()
        mcc   = s['MCC'].mean()
        acc   = s['Accuracy'].mean()
        note  = 'at chance ✓' if auroc < 0.55 else '⚠ ABOVE CHANCE — investigate'
        print(f'{target:<15} {cond:<45} {auroc:>7.4f} {mcc:>7.4f} {acc:>7.4f}  {note}')
    print()

Target          Condition                                       AUROC     MCC     Acc  Note
------------------------------------------------------------------------------------------
R/S_class       exp_0_neg_control_no_stereo_singletask         0.5000  0.0000  0.5000  at chance ✓
R/S_class       exp_0_neg_control_no_stereo_multitask          0.5000  0.0000  0.5000  at chance ✓

@/@@_class      exp_0_neg_control_no_stereo_singletask         0.5000  0.0000  0.5000  at chance ✓
@/@@_class      exp_0_neg_control_no_stereo_multitask          0.5000  0.0000  0.5000  at chance ✓

F/L_class       exp_0_neg_control_no_stereo_singletask         0.5000  0.0000  0.5000  at chance ✓
F/L_class       exp_0_neg_control_no_stereo_multitask          0.5000  0.0000  0.5000  at chance ✓



## Early stopping epoch distribution

In [15]:
epoch_summary = (
    results_df[results_df['split'] == 'test']
    .groupby(['featurization', 'target'])['stopped_epoch']
    .agg(['mean', 'min', 'max'])
    .round(1)
)
print('Epochs trained before early stopping:')
print(epoch_summary.to_string())

Epochs trained before early stopping:
                                                   mean  min  max
featurization                          target                    
exp_0_neg_control_no_stereo_multitask  @/@@_class  14.0   11   17
                                       F/L_class   14.0   11   17
                                       R/S_class   14.0   11   17
exp_0_neg_control_no_stereo_singletask @/@@_class  16.0   13   21
                                       F/L_class   14.4   12   19
                                       R/S_class   12.6   11   17


## Save results for Tukey HSD

Per-fold scores are saved to CSV for downstream statistical comparison
with Exp 1 results. The final analysis notebook loads all CSVs and runs
`scipy.stats.tukey_hsd` using the 5 per-fold test scores per condition.

In [16]:
output_path = '4_chemprop_exp_0_results.csv'
results_df.to_csv(output_path, index=False)
print(f'\nSaved {len(results_df)} total rows to {output_path}')
print(f"  Conditions: {results_df['featurization'].unique().tolist()}")
print(f"  Targets:    {results_df['target'].unique().tolist()}")
print(f"  Folds:      {sorted(results_df['fold'].unique().tolist())}")
print(f"  Splits:     {results_df['split'].unique().tolist()}")
print(f"  Columns:    {results_df.columns.tolist()}")


Saved 90 total rows to 4_chemprop_exp_0_results.csv
  Conditions: ['exp_0_neg_control_no_stereo_singletask', 'exp_0_neg_control_no_stereo_multitask']
  Targets:    ['R/S_class', '@/@@_class', 'F/L_class']
  Folds:      [0, 1, 2, 3, 4]
  Splits:     ['train', 'val', 'test']
  Columns:    ['AUROC', 'MCC', 'Accuracy', 'F1', 'AUPRC', 'fold', 'model', 'featurization', 'target', 'split', 'stopped_epoch']
